## 03_generate_embeddings.ipynb
### Purpose: Pre-compute sentence embeddings for all recipes (one-time cost)

In [5]:
# Cell 1: Imports
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
import time

print("Libraries loaded.")

Libraries loaded.


In [6]:
# Cell 2: Load cleaned data
processed_path = Path('../processed')
df = pl.read_parquet(processed_path / "recipes_cleaned.parquet")

print(f"Loaded {df.shape[0]:,} recipes")
print("Columns:", df.columns)

Loaded 231,637 recipes
Columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'ingredients_list', 'cleaned_ingredients', 'text_for_embedding']


In [7]:
# Cell 3: Load the embedding model + move to GPU if available
import torch

model_name = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

# Force GPU usage (AMD 6800M should work via DirectML or if ROCm is set up)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device.upper()}")

if device == "cuda":
    model = model.to(device)
    print("Model moved to GPU - this should be much faster!")

print(f"Model '{model_name}' loaded on {device}.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1958.95it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: CPU
Model 'all-MiniLM-L6-v2' loaded on cpu.


In [8]:
# Cell 4: Generate embeddings (GPU-accelerated)
print("Starting embedding generation...")

start_time = time.time()

embeddings = model.encode(
    df["text_for_embedding"].to_list(),
    batch_size=256,           # Larger batch size is fine on GPU
    show_progress_bar=True,
    convert_to_numpy=True,
    device=device,            # ← This is the key line
    normalize_embeddings=True # Good practice for cosine similarity later
)

end_time = time.time()
print(f"Embeddings generated in {end_time - start_time:.1f} seconds")
print("Embeddings shape:", embeddings.shape)

Starting embedding generation...


Batches: 100%|██████████| 905/905 [1:16:14<00:00,  5.05s/it]


Embeddings generated in 4585.5 seconds
Embeddings shape: (231637, 384)


In [9]:
# Cell 5: Save embeddings + keep a small index file
np.save(processed_path / "recipe_embeddings.npy", embeddings)

# Optional: Save recipe IDs for easy lookup later
recipe_ids = df.select("id").to_numpy().flatten()
np.save(processed_path / "recipe_ids.npy", recipe_ids)

print("Saved:")
print("   - recipe_embeddings.npy")
print("   - recipe_ids.npy")
print("\nYou can now move to the recommender!")

Saved:
   - recipe_embeddings.npy
   - recipe_ids.npy

You can now move to the recommender!
